In [1]:
import pandas as pd
import numpy as np

df_ml = pd.read_csv(
    r"C:\Users\DEV\Desktop\Intelligent-Helpdesk-Analytics\data\processed\tickets_ml.csv"
)

print("Shape :", df_ml.shape)

Shape : (5263, 11)


In [2]:
df_mttr = df_ml[
    df_ml["Temps_resolution_heures"].notna()
].copy()

print("Shape MTTR :", df_mttr.shape)

Shape MTTR : (5263, 11)


In [3]:
X = df_mttr[
    [
        "Priorité",
        "Categorie"
    ]
]

y = df_mttr["Temps_resolution_heures"]

print("X :", X.shape)
print("y :", y.shape)

X : (5263, 2)
y : (5263,)


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (4210, 2)
X_test  : (1053, 2)
y_train : (4210,)
y_test  : (1053,)


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            ["Priorité", "Categorie"]
        )
    ]
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("X_train après encodage :", X_train_encoded.shape)
print("X_test après encodage  :", X_test_encoded.shape)

X_train après encodage : (4210, 13)
X_test après encodage  : (1053, 13)


In [6]:
X_train_ann = X_train_encoded.toarray()
X_test_ann = X_test_encoded.toarray()

print("X_train_ann :", X_train_ann.shape)
print("X_test_ann  :", X_test_ann.shape)

X_train_ann : (4210, 13)
X_test_ann  : (1053, 13)


In [7]:
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

print("\nQuelques valeurs de y_train :")
print(y_train.head())

y_train : (4210,)
y_test  : (1053,)

Quelques valeurs de y_train :
376      50.407745
1200      4.301394
859       0.016914
4125     81.940244
2680    155.644211
Name: Temps_resolution_heures, dtype: float64


In [8]:
from sklearn.preprocessing import StandardScaler

y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(y_train.to_numpy().reshape(-1, 1))
y_test_scaled = y_scaler.transform(y_test.to_numpy().reshape(-1, 1))

print("y_train_scaled :", y_train_scaled.shape)
print("y_test_scaled  :", y_test_scaled.shape)

y_train_scaled : (4210, 1)
y_test_scaled  : (1053, 1)


In [9]:
import tensorflow as tf

print("TensorFlow :", tf.__version__)

TensorFlow : 2.21.0


In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    Dense(32, activation="relu", input_shape=(13,)),
    Dense(16, activation="relu"),
    Dense(1)
])

model.summary()

c:\Users\DEV\Desktop\Intelligent-Helpdesk-Analytics\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 993 (3.88 KB)

 Trainable params: 993 (3.88 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

print("Modèle compilé avec succès.")

Modèle compilé avec succès.


In [12]:
history = model.fit(
    X_train_ann,
    y_train_scaled,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.8746 - mae: 0.5528 - val_loss: 1.5244 - val_mae: 0.6024
Epoch 2/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8694 - mae: 0.5533 - val_loss: 1.5150 - val_mae: 0.6022
Epoch 3/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8665 - mae: 0.5563 - val_loss: 1.5175 - val_mae: 0.5950
Epoch 4/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8666 - mae: 0.5518 - val_loss: 1.5175 - val_mae: 0.5987
Epoch 5/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.8656 - mae: 0.5539 - val_loss: 1.5112 - val_mae: 0.6155
Epoch 6/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8655 - mae: 0.5593 - val_loss: 1.5149 - val_mae: 0.5998
Epoch 7/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8644 - mae: 0.5519 - val_loss: 1.5139 - val_mae: 0.6029
Epoch 8/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.8641 - mae: 0.5555 - val_loss: 1.5146 - val_mae: 0.5992
Epoch 9/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/

In [13]:
test_loss, test_mae = model.evaluate(
    X_test_ann,
    y_test_scaled,
    verbose=0
)

print("Test Loss :", test_loss)
print("Test MAE  :", test_mae)

Test Loss : 0.989361584186554
Test MAE  : 0.5758359432220459


In [14]:
# Faire les prédictions sur les données de test
y_pred_scaled = model.predict(X_test_ann)

# Revenir à l'échelle réelle (heures)
y_pred = y_scaler.inverse_transform(y_pred_scaled)

# Transformer y_test en tableau 2D pour l'inverse_transform
y_test_real = y_scaler.inverse_transform(
    y_test.to_numpy().reshape(-1, 1)
)

print("Quelques prédictions :")
for i in range(5):
    print(
        f"Réel : {y_test_real[i][0]:.2f} h"
        f" | Prédit : {y_pred[i][0]:.2f} h"
    )

33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Quelques prédictions :
Réel : 21194.82 h | Prédit : 86.67 h
Réel : 23852.47 h | Prédit : 86.67 h
Réel : 21098.74 h | Prédit : 86.67 h
Réel : 80182.43 h | Prédit : 86.67 h
Réel : 1737.13 h | Prédit : 86.67 h


In [15]:
print("Minimum :", y.min())
print("Maximum :", y.max())
print("Moyenne :", y.mean())
print("Médiane :", y.median())

print("\nQuantiles :")
print(y.quantile([0, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99, 1]))

Minimum : 0.0041886111111111
Maximum : 2349.8134247222224
Moyenne : 88.60598299364536
Médiane : 50.68773916666667

Quantiles :
0.00       0.004189
0.25      21.033604
0.50      50.687739
0.75     106.926332
0.90     197.180354
0.95     297.826661
0.99     613.234905
1.00    2349.813425
Name: Temps_resolution_heures, dtype: float64


In [16]:
print("y_test original :")
print(y_test.head())

print("\nStatistiques de y_test :")
print(y_test.describe())

y_test original :
1144    160.564841
3942    180.783287
3665    159.833878
3209    609.320346
1074     12.538066
Name: Temps_resolution_heures, dtype: float64

Statistiques de y_test :
count    1053.000000
mean       86.886155
std       130.748816
min         0.006397
25%        20.363633
50%        47.289017
75%       104.378664
max      1877.066312
Name: Temps_resolution_heures, dtype: float64


In [17]:
print("Moyenne utilisée par y_scaler :", y_scaler.mean_)
print("Écart-type utilisé :", y_scaler.scale_)

print("\nMoyenne réelle de y_train :", y_train.mean())
print("Écart-type réel de y_train :", y_train.std())

Moyenne utilisée par y_scaler : [89.03614431]
Écart-type utilisé : [131.44709789]

Moyenne réelle de y_train : 89.0361443050277
Écart-type réel de y_train : 131.46271196997853


In [18]:
print("y_test[0] :", y_test.iloc[0])
print("y_test[1] :", y_test.iloc[1])

print("\ny_test_scaled[0] :", y_test_scaled[0])
print("y_test_scaled[1] :", y_test_scaled[1])

print("\ninverse_transform :")
print(y_scaler.inverse_transform(y_test_scaled[:2]))

y_test[0] : 160.56484083333334
y_test[1] : 180.78328722222224

y_test_scaled[0] : [0.54416338]
y_test_scaled[1] : [0.6979777]

inverse_transform :
[[160.56484083]
 [180.78328722]]


In [19]:
# 1. Prédictions du modèle sur les données de test
y_pred_scaled = model.predict(X_test_ann, verbose=0)

# 2. Conversion des prédictions vers les heures réelles
y_pred = y_scaler.inverse_transform(y_pred_scaled).ravel()

# 3. Récupération des vraies valeurs en heures
y_test_real = y_test.to_numpy()

# 4. Affichage des premières prédictions
print("Quelques prédictions :")

for i in range(5):
    print(
        f"Réel : {y_test_real[i]:.2f} h"
        f" | Prédit : {y_pred[i]:.2f} h"
    )

Quelques prédictions :
Réel : 160.56 h | Prédit : 86.67 h
Réel : 180.78 h | Prédit : 86.67 h
Réel : 159.83 h | Prédit : 86.67 h
Réel : 609.32 h | Prédit : 86.67 h
Réel : 12.54 h | Prédit : 86.67 h


In [20]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test_real, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred))
r2 = r2_score(y_test_real, y_pred)

print("MAE :", mae, "heures")
print("RMSE :", rmse, "heures")
print("R² :", r2)

MAE : 75.69196123155665 heures
RMSE : 130.74602899485112 heures
R² : -0.0009079002843461748


In [21]:
print(df_ml.columns.tolist())

['ticket_id', 'État', 'ID de phase', 'Priorité', 'Titre de la demande', 'Demandé pour Nom', 'Date MAJ', 'Date de résolution', 'Date/Heure de création', 'Categorie', 'Temps_resolution_heures']


In [22]:
print(df_ml["Date/Heure de création"].dtype)

str


In [23]:
df_ml["Date/Heure de création"] = pd.to_datetime(
    df_ml["Date/Heure de création"],
    errors="coerce"
)

print(df_ml["Date/Heure de création"].dtype)

datetime64[us]


In [24]:
df_ml["heure_creation"] = df_ml["Date/Heure de création"].dt.hour
df_ml["jour_semaine"] = df_ml["Date/Heure de création"].dt.dayofweek
df_ml["mois_creation"] = df_ml["Date/Heure de création"].dt.month
df_ml["jour_mois"] = df_ml["Date/Heure de création"].dt.day

print(
    df_ml[
        [
            "Date/Heure de création",
            "heure_creation",
            "jour_semaine",
            "mois_creation",
            "jour_mois"
        ]
    ].head()
)

   Date/Heure de création  heure_creation  jour_semaine  mois_creation  \
0 2026-02-10 13:55:26.031              13             1              2   
1 2026-02-06 16:08:36.542              16             4              2   
2 2026-02-06 16:06:12.849              16             4              2   
3 2025-11-29 19:31:37.933              19             5             11   
4 2026-02-10 13:58:45.989              13             1              2   

   jour_mois  
0         10  
1          6  
2          6  
3         29  
4         10  


In [25]:
print(
    df_ml[
        [
            "Priorité",
            "Categorie",
            "heure_creation",
            "jour_semaine",
            "mois_creation",
            "jour_mois",
            "Temps_resolution_heures"
        ]
    ].head()
)

      Priorité                        Categorie  heure_creation  jour_semaine  \
0  Faible (P4)  Assistance & Demandes generales              13             1   
1  Faible (P4)  Assistance & Demandes generales              16             4   
2  Faible (P4)  Assistance & Demandes generales              16             4   
3  Faible (P4)  Assistance & Demandes generales              19             5   
4  Faible (P4)  Assistance & Demandes generales              13             1   

   mois_creation  jour_mois  Temps_resolution_heures  
0              2         10                72.414991  
1              2          6               172.086516  
2              2          6               325.951153  
3             11         29               152.364185  
4              2         10                12.077503  


In [26]:
features_v2 = [
    "Priorité",
    "Categorie",
    "Titre de la demande",
    "heure_creation",
    "jour_semaine",
    "mois_creation",
    "jour_mois"
]

X_v2 = df_ml[features_v2]
y_v2 = df_ml["Temps_resolution_heures"]

print("X_v2 :", X_v2.shape)
print("y_v2 :", y_v2.shape)

X_v2 : (5263, 7)
y_v2 : (5263,)


In [27]:
X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
    X_v2,
    y_v2,
    test_size=0.2,
    random_state=42
)

print("X_train_v2 :", X_train_v2.shape)
print("X_test_v2  :", X_test_v2.shape)
print("y_train_v2 :", y_train_v2.shape)
print("y_test_v2  :", y_test_v2.shape)

X_train_v2 : (4210, 7)
X_test_v2  : (1053, 7)
y_train_v2 : (4210,)
y_test_v2  : (1053,)


In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    lowercase=True,
    max_features=50
)

X_train_tfidf = tfidf.fit_transform(
    X_train_v2["Titre de la demande"]
)

X_test_tfidf = tfidf.transform(
    X_test_v2["Titre de la demande"]
)

print("X_train TF-IDF :", X_train_tfidf.shape)
print("X_test TF-IDF  :", X_test_tfidf.shape)

X_train TF-IDF : (4210, 50)
X_test TF-IDF  : (1053, 50)


In [29]:
print(df_ml["Titre de la demande"].head())
print("\nValeurs manquantes :", df_ml["Titre de la demande"].isna().sum())

0    Demande de service
1    Demande de service
2    Demande de service
3    Demande de service
4    Demande de service
Name: Titre de la demande, dtype: str

Valeurs manquantes : 0


In [30]:
print("Nombre de titres différents :", df_ml["Titre de la demande"].nunique())

print("\nTitres les plus fréquents :")
print(df_ml["Titre de la demande"].value_counts().head(20))

Nombre de titres différents : 96

Titres les plus fréquents :
Titre de la demande
Demande de service                                                                            4094
Demande escaladee - assistance requise                                                         751
Activation compte AD                                                                           165
Assistance support niveau 1                                                                     45
Alerte sécurité poste de travail                                                                40
Assistance                                                                                      39
Demande de réactivation session windows & Demande de réinitialisation mot de passe session      16
Assistance SAP                                                                                   9
Demande de réinitialisation mot de passe session windows                                         5
demande de réinitialisation

In [31]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = ["Priorité", "Categorie"]

preprocessor_cat_v2 = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="drop"
)

X_train_cat_v2 = preprocessor_cat_v2.fit_transform(
    X_train_v2
)

X_test_cat_v2 = preprocessor_cat_v2.transform(
    X_test_v2
)

print("Catégories train :", X_train_cat_v2.shape)
print("Catégories test  :", X_test_cat_v2.shape)

Catégories train : (4210, 13)
Catégories test  : (1053, 13)


In [32]:
X_train_time_v2 = X_train_v2[
    [
        "heure_creation",
        "jour_semaine",
        "mois_creation",
        "jour_mois"
    ]
].to_numpy()

X_test_time_v2 = X_test_v2[
    [
        "heure_creation",
        "jour_semaine",
        "mois_creation",
        "jour_mois"
    ]
].to_numpy()

print("Temps train :", X_train_time_v2.shape)
print("Temps test  :", X_test_time_v2.shape)

Temps train : (4210, 4)
Temps test  : (1053, 4)


In [33]:
from scipy.sparse import hstack

X_train_v2_final = hstack([
    X_train_cat_v2,
    X_train_time_v2,
    X_train_tfidf
])

X_test_v2_final = hstack([
    X_test_cat_v2,
    X_test_time_v2,
    X_test_tfidf
])

print("X_train_v2_final :", X_train_v2_final.shape)
print("X_test_v2_final  :", X_test_v2_final.shape)

X_train_v2_final : (4210, 67)
X_test_v2_final  : (1053, 67)


In [34]:
from sklearn.preprocessing import StandardScaler

y_scaler_v2 = StandardScaler()

y_train_v2_scaled = y_scaler_v2.fit_transform(
    y_train_v2.to_numpy().reshape(-1, 1)
)

y_test_v2_scaled = y_scaler_v2.transform(
    y_test_v2.to_numpy().reshape(-1, 1)
)

print("y_train_v2_scaled :", y_train_v2_scaled.shape)
print("y_test_v2_scaled  :", y_test_v2_scaled.shape)

y_train_v2_scaled : (4210, 1)
y_test_v2_scaled  : (1053, 1)


In [35]:
X_train_v2_ann = X_train_v2_final.toarray()
X_test_v2_ann = X_test_v2_final.toarray()

print("X_train_v2_ann :", X_train_v2_ann.shape)
print("X_test_v2_ann  :", X_test_v2_ann.shape)

X_train_v2_ann : (4210, 67)
X_test_v2_ann  : (1053, 67)


In [36]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout

model_v2 = Sequential([
    Input(shape=(67,)),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(1, activation="linear")
])

model_v2.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model_v2.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │         4,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,977 (27.25 KB)

 Trainable params: 6,977 (27.25 KB)

 Non-trainable params: 0 (0.00 B)

In [37]:
history_v2 = model_v2.fit(
    X_train_v2_ann,
    y_train_v2_scaled,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    verbose=1
)

Epoch 1/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 1.1270 - mae: 0.6732 - val_loss: 1.5290 - val_mae: 0.6113
Epoch 2/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.8831 - mae: 0.5635 - val_loss: 1.5235 - val_mae: 0.5918
Epoch 3/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8775 - mae: 0.5595 - val_loss: 1.5198 - val_mae: 0.5925
Epoch 4/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8773 - mae: 0.5544 - val_loss: 1.5154 - val_mae: 0.5989
Epoch 5/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8716 - mae: 0.5557 - val_loss: 1.5138 - val_mae: 0.6032
Epoch 6/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8728 - mae: 0.5574 - val_loss: 1.5153 - val_mae: 0.6021
Epoch 7/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8739 - mae: 0.5597 - val_loss: 1.5150 - val_mae: 0.6035
Epoch 8/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8746 - mae: 0.5622 - val_loss: 1.5143 - val_mae: 0.6044
Epoch 9/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/

In [38]:
test_loss_v2, test_mae_v2 = model_v2.evaluate(
    X_test_v2_ann,
    y_test_v2_scaled,
    verbose=1
)

print("V2 Test Loss :", test_loss_v2)
print("V2 Test MAE  :", test_mae_v2)

33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.9916 - mae: 0.5730
V2 Test Loss : 0.9916178584098816
V2 Test MAE  : 0.5730221271514893


In [39]:
y_pred_v2_scaled = model_v2.predict(X_test_v2_ann)

y_pred_v2 = y_scaler_v2.inverse_transform(
    y_pred_v2_scaled
)

print("Quelques prédictions V2 :")

for i in range(5):
    print(
        f"Réel : {y_test_v2.iloc[i]:.2f} h | "
        f"Prédit : {y_pred_v2[i][0]:.2f} h"
    )

33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Quelques prédictions V2 :
Réel : 160.56 h | Prédit : 93.70 h
Réel : 180.78 h | Prédit : 94.73 h
Réel : 159.83 h | Prédit : 82.73 h
Réel : 609.32 h | Prédit : 85.62 h
Réel : 12.54 h | Prédit : 88.62 h


In [41]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred_v2_flat = y_pred_v2.flatten()
y_test_v2_flat = y_test_v2.to_numpy()

mae_v2 = mean_absolute_error(
    y_test_v2_flat,
    y_pred_v2_flat
)

rmse_v2 = np.sqrt(
    mean_squared_error(
        y_test_v2_flat,
        y_pred_v2_flat
    )
)

r2_v2 = r2_score(
    y_test_v2_flat,
    y_pred_v2_flat
)

print("MAE V2 :", mae_v2, "heures")
print("RMSE V2 :", rmse_v2, "heures")
print("R² V2 :", r2_v2)

MAE V2 : 75.32208463183017 heures
RMSE V2 : 130.89503732082292 heures
R² V2 : -0.0031906250391977586


In [43]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train_v2_final,
    y_train_v2
)

print("Random Forest entraîné avec succès.")

Random Forest entraîné avec succès.


In [44]:
y_pred_rf = rf_model.predict(X_test_v2_final)

print("Quelques prédictions Random Forest :")

for i in range(5):
    print(
        f"Réel : {y_test_v2.iloc[i]:.2f} h | "
        f"Prédit : {y_pred_rf[i]:.2f} h"
    )

Quelques prédictions Random Forest :
Réel : 160.56 h | Prédit : 111.69 h
Réel : 180.78 h | Prédit : 97.08 h
Réel : 159.83 h | Prédit : 111.35 h
Réel : 609.32 h | Prédit : 85.50 h
Réel : 12.54 h | Prédit : 91.28 h


In [45]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae_rf = mean_absolute_error(y_test_v2, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test_v2, y_pred_rf))
r2_rf = r2_score(y_test_v2, y_pred_rf)

print(f"MAE Random Forest  : {mae_rf:.2f} heures")
print(f"RMSE Random Forest : {rmse_rf:.2f} heures")
print(f"R² Random Forest   : {r2_rf:.4f}")

MAE Random Forest  : 79.76 heures
RMSE Random Forest : 137.23 heures
R² Random Forest   : -0.1026


In [46]:
import numpy as np

y_train_v3_log = np.log1p(y_train_v2)

print("Quelques valeurs originales :")
print(y_train_v2.head())

print("\nQuelques valeurs après log1p :")
print(y_train_v3_log.head())

Quelques valeurs originales :
376      50.407745
1200      4.301394
859       0.016914
4125     81.940244
2680    155.644211
Name: Temps_resolution_heures, dtype: float64

Quelques valeurs après log1p :
376     3.939789
1200    1.667970
859     0.016773
4125    4.418120
2680    5.053977
Name: Temps_resolution_heures, dtype: float64


In [47]:
rf_model_v3 = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model_v3.fit(
    X_train_v2_final,
    y_train_v3_log
)

print("Random Forest V3 entraîné avec succès.")

Random Forest V3 entraîné avec succès.


In [48]:
y_pred_v3_log = rf_model_v3.predict(X_test_v2_final)

# Retour aux heures réelles
y_pred_v3 = np.expm1(y_pred_v3_log)

print("Quelques prédictions Random Forest V3 :")

for i in range(5):
    print(
        f"Réel : {y_test_v2.iloc[i]:.2f} h | "
        f"Prédit : {y_pred_v3[i]:.2f} h"
    )

Quelques prédictions Random Forest V3 :
Réel : 160.56 h | Prédit : 103.21 h
Réel : 180.78 h | Prédit : 88.19 h
Réel : 159.83 h | Prédit : 75.07 h
Réel : 609.32 h | Prédit : 84.47 h
Réel : 12.54 h | Prédit : 69.15 h


In [49]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae_rf_v3 = mean_absolute_error(y_test_v2, y_pred_v3)
rmse_rf_v3 = np.sqrt(mean_squared_error(y_test_v2, y_pred_v3))
r2_rf_v3 = r2_score(y_test_v2, y_pred_v3)

print(f"MAE Random Forest V3  : {mae_rf_v3:.2f} heures")
print(f"RMSE Random Forest V3 : {rmse_rf_v3:.2f} heures")
print(f"R² Random Forest V3   : {r2_rf_v3:.4f}")

MAE Random Forest V3  : 67.75 heures
RMSE Random Forest V3 : 135.90 heures
R² Random Forest V3   : -0.0814


In [51]:
import xgboost as xgb

print("XGBoost :", xgb.__version__)

XGBoost : 3.4.1


In [52]:
print("Train :", X_train_v2_final.shape)
print("Test  :", X_test_v2_final.shape)

Train : (4210, 67)
Test  : (1053, 67)


In [53]:
import numpy as np

y_train_xgb_log = np.log1p(y_train_v2)

print("Quelques valeurs originales :")
print(y_train_v2.head())

print("\nQuelques valeurs après log1p :")
print(y_train_xgb_log.head())

Quelques valeurs originales :
376      50.407745
1200      4.301394
859       0.016914
4125     81.940244
2680    155.644211
Name: Temps_resolution_heures, dtype: float64

Quelques valeurs après log1p :
376     3.939789
1200    1.667970
859     0.016773
4125    4.418120
2680    5.053977
Name: Temps_resolution_heures, dtype: float64


In [54]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    objective="reg:squarederror"
)

xgb_model.fit(
    X_train_v2_final,
    y_train_xgb_log
)

print("XGBoost entraîné avec succès.")

XGBoost entraîné avec succès.


In [55]:
y_pred_xgb_log = xgb_model.predict(X_test_v2_final)

# Retour aux heures réelles
y_pred_xgb = np.expm1(y_pred_xgb_log)

print("Quelques prédictions XGBoost :")

for i in range(5):
    print(
        f"Réel : {y_test_v2.iloc[i]:.2f} h | "
        f"Prédit : {y_pred_xgb[i]:.2f} h"
    )

Quelques prédictions XGBoost :
Réel : 160.56 h | Prédit : 60.34 h
Réel : 180.78 h | Prédit : 69.72 h
Réel : 159.83 h | Prédit : 96.07 h
Réel : 609.32 h | Prédit : 76.74 h
Réel : 12.54 h | Prédit : 64.36 h


In [56]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae_xgb = mean_absolute_error(y_test_v2, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test_v2, y_pred_xgb))
r2_xgb = r2_score(y_test_v2, y_pred_xgb)

print(f"MAE XGBoost  : {mae_xgb:.2f} heures")
print(f"RMSE XGBoost : {rmse_xgb:.2f} heures")
print(f"R² XGBoost   : {r2_xgb:.4f}")

MAE XGBoost  : 67.19 heures
RMSE XGBoost : 135.57 heures
R² XGBoost   : -0.0761
